# Example 1: BHLUMI + WarpX, round head-on beams vs analytic theory

This is the first validation case for injecting BHLUMI Bhabha events into WarpX. The beams are chosen so the
beam-beam deflection of the outgoing e⁺e⁻ has a closed-form prediction:

- round Gaussian beams (σ_x = σ_y = σ), head-on, no crossing angle
- rigid bunches (disruption D ≪ 1), no hourglass (zero emittance), no energy spread
- no QED (no beamstrahlung), no external fields

**One source of truth.** Every number the theory uses is read back from the WarpX input file. Either edit
`PARAMS` below and let the notebook write the file, or edit the file by hand and set `WRITE_INPUTS = False`.
The validity checks and the predictions always describe the file that will be run.

Workflow:
1. Set the parameters and write the input file (Section 1).
2. Check that the file satisfies the assumptions of the formula (Section 2).
3. Look at the predictions (Section 3).
4. Run on Perlmutter with `benchmarks/submit_benchmark.sh` (Section 4).
5. Point `RUN_DIR` at the run and compare (Section 5).

## Theory

**Setup.** Two round Gaussian bunches with N particles each, rms size σ transversely and σ_z longitudinally,
collide head-on. A Bhabha e⁻ (from beam1, moving along +z) leaves its vertex at polar angle θ and azimuth φ.
It then crosses the rest of the e⁺ bunch, which attracts it toward the axis.

**Assumptions:** rigid bunches, β* ≫ σ_z, no beamstrahlung, ultrarelativistic lepton with θ ≪ 1, kick small
enough not to change the path, and only the opposite bunch acts (the lepton's own bunch moves almost parallel
to it, so its E and B forces cancel to order θ²/4).

**Step 1. Field of the opposite bunch.** The e⁺ bunch moves at −c with line density
λ(ζ) = N e g(ζ), g(ζ) = exp(−ζ²/2σ_z²)/(√(2π) σ_z). Its field is purely transverse:

$$\mathbf E_\perp(\mathbf r,\zeta)=\frac{\lambda(\zeta)}{2\pi\varepsilon_0}\frac{\mathbf r}{r^2}\left(1-e^{-r^2/2\sigma^2}\right),\qquad c\mathbf B_\perp=\hat z\times\mathbf E_\perp .$$

**Step 2. Force on a counter-moving lepton.** Electric and magnetic forces add: F⊥ = q(E + v×B)⊥ = 2qE⊥.

**Step 3. Path.** Lepton and bunch approach at 2c. After the lepton has crossed a length u = ζ − ζ₀ of the
bunch, t = u/2c and its transverse position is **r**(u) = **r**₀ + n̂ θu/2, with n̂ = (cos φ, sin φ).

**Step 4. Kick for one event.** Integrate the force, project on n̂, and divide by p = γ′mc, using
r_e = e²/(4πε₀mc²):

$$\Delta\theta=-\frac{2Nr_e}{\gamma'}\int_0^\infty g(\zeta_0+u)\,\frac{\mathbf r\cdot\hat n}{r^2}\left(1-e^{-r^2/2\sigma^2}\right)du .$$

**Step 5. Average over the transverse vertex.** The luminosity density is the product of the two beam
densities, so **r**₀ is Gaussian with variance σ²/2 per axis. The field is linear in the charge, so averaging
the field point over that offset is the field of a Gaussian of variance σ²_eff = σ² + σ²/2 = 3σ²/2, taken at
**d** = n̂θu/2:

$$\left\langle\frac{\mathbf r\cdot\hat n}{r^2}\left(1-e^{-r^2/2\sigma^2}\right)\right\rangle_{\mathbf r_0}=\frac{2}{\theta u}\left(1-e^{-\theta^2u^2/12\sigma^2}\right).$$

**Step 6. Average over the longitudinal vertex.** Without hourglass, ζ₀ ~ g independently of **r**₀, and
∫ g(ζ₀) g(ζ₀+u) dζ₀ = G(u) = exp(−u²/4σ_z²)/(2√π σ_z). So

$$\langle\Delta\theta\rangle=-\frac{4Nr_e}{\gamma'\theta}\frac{1}{2\sqrt\pi\sigma_z}\int_0^\infty\frac{e^{-bu^2}-e^{-(a+b)u^2}}{u}\,du,\qquad a=\frac{\theta^2}{12\sigma^2},\ b=\frac{1}{4\sigma_z^2}.$$

**Step 7. Frullani integral.** The integral is ½ ln((a+b)/b), which gives

$$\boxed{\langle\Delta\theta\rangle(\theta)=-\frac{N r_e}{\sqrt{\pi}\,\gamma'\,\theta\,\sigma_z}\ln\!\left(1+\frac{\theta^2\sigma_z^2}{3\sigma^2}\right)}$$

The same holds for the e⁺ (θ measured from −z). Limits: θσ_z ≪ σ gives a thin-lens kick
−N r_e θ σ_z/(3√π γ′σ²) ∝ θ; θσ_z ≫ σ gives the line-charge result ∝ ln θ/θ. For round beams the kick does not
depend on φ.

`bhabha_theory.kick_mc` computes the same average numerically and can add what the formula leaves out: the box
walls (the lepton stops feeling the field when it is absorbed), the end of the run, the bunches' z cut, and
σ_x ≠ σ_y.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "benchmarks"))
import bhabha_theory as bt
import make_inputs as mi

## 1. Parameters and input file

`PARAMS` overrides `make_inputs.HEADON_DEFAULTS` (printed below). With `WRITE_INPUTS = True` the notebook writes
`INPUT_FILE` from them. With `False` it only reads `INPUT_FILE`, e.g. a file you edited or one from
`make_inputs.ipynb`.

In [ ]:
print("defaults:", mi.HEADON_DEFAULTS)

PARAMS = {
    "E_GeV": 45.6,
    "npart": 2.0e10,
    "sigma_um": 10.0,
    "sigmaz_mm": 1.0,
    "nx": 512, "ny": 512, "nz": 1024,
}
WRITE_INPUTS = True
INPUT_FILE = os.path.join(REPO, "benchmarks", "headon_round", "inputs_headon_round.txt")

if WRITE_INPUTS:
    mi.write(mi.headon_round_inputs(PARAMS), INPUT_FILE)
p = bt.Params(INPUT_FILE)
bt.print_table(p.table())

## 2. Does the file satisfy the formula's assumptions?

The angle range should match the BHLUMI sample (the FCC-ee Z card in this repo generates 40–120 mrad). If
`EVENTS` exists, the range is taken from it. Any `False` row means the comparison in Section 5 tests something
other than the formula.

In [ ]:
EVENTS = os.path.expandvars("$PSCRATCH/bhlumi/fcc_z/seed_2/events.npz")
THETA_MIN, THETA_MAX = 0.040, 0.120
if os.path.isfile(EVENTS):
    ev = np.load(EVENTS)
    th = np.arctan2(np.hypot(ev["ele"][:, 0], ev["ele"][:, 1]), -ev["ele"][:, 2])   # BHLUMI e- beam along -z
    THETA_MIN, THETA_MAX = np.percentile(th, [0.5, 99.5])
    print(f"theta range from {EVENTS}: {THETA_MIN*1e3:.1f}-{THETA_MAX*1e3:.1f} mrad, sigma = {float(ev['xsec_nb']):.5g} nb")
checks = bt.validity_checks(p, THETA_MIN, THETA_MAX)
bt.print_table(checks, ("check", "value", "requirement", "ok"))
print("\nall checks pass" if all(c[3] for c in checks) else "\nSOME CHECKS FAIL -- fix the inputs before running")

## 3. Predictions

In [ ]:
L = bt.luminosity_per_crossing(p)
print(f"luminosity per crossing N^2/(4 pi sx sy) = {L:.4g} m^-2 = {L*1e-4:.4g} cm^-2")
print(f"disruption D = {bt.disruption(p):.3g}")
if os.path.isfile(EVENTS):
    print(f"Bhabhas per crossing in the sample's acceptance: {float(ev['xsec_nb'])*1e-37*L:.3g}")

theta = np.linspace(THETA_MIN, THETA_MAX, 200)
th_mc = np.linspace(THETA_MIN, THETA_MAX, 5)
mc = np.array([bt.kick_mc(t, p, n_events=10000, finite_box=True) for t in th_mc])

print("\n theta [mrad]   closed form [urad]   numerical, with box [urad]")
for t, (m, e, _) in zip(th_mc, mc):
    print(f"   {t*1e3:6.1f}        {bt.kick_closed_form(t, p)*1e6:9.3f}          {m*1e6:9.3f} +- {e*1e6:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(theta*1e3, bt.kick_closed_form(theta, p)*1e6, label="closed form")
ax.errorbar(th_mc*1e3, mc[:, 0]*1e6, mc[:, 1]*1e6, fmt="o", label="numerical average, finite box")
ax.set_xlabel(r"$\theta$ [mrad]"); ax.set_ylabel(r"$\langle\Delta\theta\rangle$ [$\mu$rad]")
ax.set_title(f"N={p.N:.3g}, $\\sigma$={p.sigma*1e6:.3g} $\\mu$m, $\\sigma_z$={p.sz*1e3:.3g} mm, E={p.E_GeV:.4g} GeV")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Run on Perlmutter

`benchmarks/submit_benchmark.sh` runs `wx_bhabha.py --tag-weights` on this input (no `Bz` override). The cell
prints the command for the current `INPUT_FILE`, with the path as it is inside your Perlmutter clone.

In [ ]:
PERLMUTTER_REPO = "$HOME/src/bhlumi-warpx"
TAG, SEED, NINJ = "headon_round", 1, 100000
rel = os.path.relpath(INPUT_FILE, REPO)
inputs_on_nersc = f"{PERLMUTTER_REPO}/{rel}" if not rel.startswith("..") else INPUT_FILE
print(f"cd {PERLMUTTER_REPO} && git pull")
print(f"sbatch -q debug -t 00:15:00 --export=ALL,INPUTS={inputs_on_nersc},TAG={TAG}_test,SEED={SEED},NINJ=5000 benchmarks/submit_benchmark.sh")
print(f"sbatch --export=ALL,INPUTS={inputs_on_nersc},TAG={TAG},SEED={SEED},NINJ={NINJ} benchmarks/submit_benchmark.sh")
print(f"\nrun folder: $PSCRATCH/runs/warpx/bhabha_benchmark/{TAG}/seed_{SEED}")

## 5. Compare the run with the theory

Needs `openpmd_api` (in the WarpX environment). Each lepton's Δθ = θ_out − θ₀ is compared with the closed form
evaluated at its own θ₀ and energy E₀ (the kick scales as 1/γ′, so BHLUMI leptons that radiated photons get
larger kicks). The pull column is (simulation − theory)/error per bin.

In [ ]:
RUN_DIR = os.path.expandvars(os.environ.get("BENCH_RUN_DIR", f"$PSCRATCH/runs/warpx/bhabha_benchmark/{TAG}/seed_{SEED}"))
run = None
if not os.path.isdir(RUN_DIR):
    print(f"no run at {RUN_DIR} yet")
else:
    pr = bt.Params(os.path.join(RUN_DIR, "inputs_template.txt")) if os.path.isfile(os.path.join(RUN_DIR, "inputs_template.txt")) else p
    if pr.raw != p.raw:
        print("NOTE: the run used a different input file than INPUT_FILE; comparing with the run's own file")
    run = bt.load_run(RUN_DIR)
    n_out = len(run["ievt"])
    print(f"{run['n_injected']} events injected, {n_out} leptons found "
          f"({(run['where']==0).sum()} at walls, {(run['where']==1).sum()} still in the box)")
    if n_out < 2 * run["n_injected"]:
        print(f"WARNING: {2*run['n_injected'] - n_out} leptons missing (injected after the run ended?)")
    Lsim = bt.luminosity_from_reduced(RUN_DIR, pr)
    if Lsim is not None:
        Lth = bt.luminosity_per_crossing(pr)
        print(f"luminosity: WarpX {Lsim:.4g}, theory {Lth:.4g} m^-2 (ratio {Lsim/Lth:.4f})")
    bins = np.linspace(THETA_MIN, THETA_MAX, 9)
    for sp, name in ((0, "e-"), (1, "e+"), (None, "both")):
        sel = np.ones(n_out, bool) if sp is None else run["species"] == sp
        sub = {k: (v[sel] if isinstance(v, np.ndarray) and v.shape[:1] == (n_out,) else v) for k, v in run.items()}
        rows, arr, (res, err) = bt.compare(sub, pr, bins)
        print(f"\n{name}: mean residual (sim - theory) = {res*1e6:.3f} +- {err*1e6:.3f} urad")
        bt.print_table(rows, ("theta [mrad]", "n", "sim [urad]", "theory [urad]", "pull"))

In [ ]:
if run is not None:
    rows, arr, _ = bt.compare(run, pr, bins)
    pred = bt.kick_closed_form(run["theta0"], pr, E_lepton_GeV=run["E0"])
    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].errorbar(arr["theta"]*1e3, arr["sim"]*1e6, arr["err"]*1e6, fmt="o", label="WarpX")
    axs[0].plot(arr["theta"]*1e3, arr["theory"]*1e6, "s", mfc="none", label="theory (same events)")
    axs[0].plot(theta*1e3, bt.kick_closed_form(theta, pr)*1e6, "-", lw=0.8, label="closed form, E = E_beam")
    axs[0].set_xlabel(r"$\theta_0$ [mrad]"); axs[0].set_ylabel(r"$\langle\Delta\theta\rangle$ [$\mu$rad]"); axs[0].legend()
    ph = np.mod(run["phi0"], 2*np.pi); pb = np.linspace(0, 2*np.pi, 13)
    ratio = [np.mean(run["dtheta"][(ph>=a)&(ph<b)]) / np.mean(pred[(ph>=a)&(ph<b)]) for a, b in zip(pb[:-1], pb[1:])]
    axs[1].plot(0.5*(pb[:-1]+pb[1:]), ratio, "o"); axs[1].axhline(1, color="k", lw=0.8)
    axs[1].set_xlabel(r"$\phi_0$ [rad]"); axs[1].set_ylabel("sim / theory"); axs[1].set_title("flat for round beams")
    axs[2].hist((run["dtheta"] - pred)*1e6, bins=80)
    axs[2].set_xlabel(r"$\Delta\theta$ − theory [$\mu$rad] (per lepton)"); axs[2].set_title("vertex-to-vertex spread")
    plt.tight_layout(); plt.show()